<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/Copy_of_2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [23]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [24]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [25]:
print('rows before:', len(df), '| exact duplicate rows:', df.duplicated().sum())
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log('dedupe', 'drop exact duplicate rows (injected by df.sample)', before - len(df))
assert df['order_id'].is_unique

rows before: 315 | exact duplicate rows: 15
[dedupe] drop exact duplicate rows (injected by df.sample) (15 row(s))


### TODO 2 — clean `price` -> float

In [26]:
df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)
log('price', "strip '$' and cast price to float", len(df))

[price] strip '$' and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [27]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
before = len(df)
df = df[df['qty'].notna() & (df['qty'] >= 1)].copy()
df['qty'] = df['qty'].astype(int)
log('qty', 'coerce qty to numeric; drop rows with missing or negative qty', before - len(df))

[qty] coerce qty to numeric; drop rows with missing or negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [28]:
print(df['item'].value_counts())
ITEM_MAP = {'Cheeseburger': 'Cheeseburger', 'cheese burger': 'Cheeseburger',
            'Foam Finger': 'Foam Finger', 'foam finger': 'Foam Finger',
            'Rain Poncho': 'Rain Poncho', 'rain poncho': 'Rain Poncho'}
before = df['item'].nunique()
changed = (df['item'] != df['item'].map(ITEM_MAP)).sum()
df['item'] = df['item'].map(ITEM_MAP)
log('item', f'map {before} spellings to {df["item"].nunique()} products', changed)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] map 6 spellings to 3 products (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [29]:
CAT_MAP = {'Food': 'Food', 'food': 'Food', 'Merch': 'Merch',
           'Apparel': 'Merch',   # business decision
           'RainGear': 'RainGear', 'rain-gear': 'RainGear'}
changed = (df['category'] != df['category'].map(CAT_MAP)).sum()
n_apparel = (df['category'] == 'Apparel').sum()
df['category'] = df['category'].map(CAT_MAP)
log('category', f'normalize case/spelling; fold Apparel into Merch ({n_apparel} Apparel rows)', changed)

[category] normalize case/spelling; fold Apparel into Merch (43 Apparel rows) (132 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [30]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item']) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category']) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [31]:
df['revenue'] = (df['qty'] * df['price']).round(2)
print(df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2))
print('TOTAL:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
TOTAL: 4740.0


**What I would tell the vendor:** _Food is the top category at $1,656.00 of $4,740.00 total, so I would stock more food, though its lead over Merch and RainGear is pretty small._

### TODO 8 — read back your log

In [32]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dedupe,drop exact duplicate rows (injected by df.sample),15
1,price,strip '$' and cast price to float,300
2,qty,coerce qty to numeric; drop rows with missing ...,25
3,item,map 6 spellings to 3 products,126
4,category,normalize case/spelling; fold Apparel into Mer...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

Dropping duplicates changed revenue the most. The total went from $4,852.50 before to $4,594.50 after, a drop of $258.00 from 15 duplicate rows. The qty step moved it less, from $4,594.50 to $4,740.00 (+$145.50). All of that came from removing negative quantities. The rows with missing qty added nothing either way.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

I think the most debatable choice was putting Apparel under Merch. Someone else could have kept Apparel as its own category, and then the numbers would be Food $1,656.00, RainGear $1,512.00, Merch $856.50, and Apparel $715.50. The total revenue stays at $4,740.00 either way, but Merch would fall from second to third and look like it isn't selling well, when really its sales are just split across two labels. I combined them because a vendor would probably think of foam fingers and other merchandise as one group when deciding what to stock, and the difference between "Apparel" and "Merch" seemed more like inconsistent labeling than two different kinds of products.